# Cal3_DS2

The `Cal3_DS2` class represents a camera calibration model that includes intrinsic parameters and two tangential distortion coefficients. This model is particularly useful for cameras where lens distortion has a significant tangential component.

It includes the standard 5 intrinsic parameters—two focal lengths ($f_x, f_y$), skew ($s$), and the principal point ($u_0, v_0$)—plus two parameters for tangential distortion ($p_1, p_2$). The total number of parameters is 8 (fx, fy, s, u0, v0, k1, k2, p1, p2), but since `Cal3_DS2` does not model radial distortion, k1 and k2 are held at zero.

The calibration matrix $K$ is the same as in the simpler models:
$$
K = \begin{bmatrix} f_x & s & u_0 \\ 0 & f_y & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

The key feature of `Cal3_DS2` is its distortion model. Given a point in normalized image coordinates $(x_u, y_u)$, the tangential distortion is computed as:
$$
dx = 2 p_1 x_u y_u + p_2 (r^2 + 2 x_u^2) \\
dy = p_1 (r^2 + 2 y_u^2) + 2 p_2 x_u y_u
$$
where $r^2 = x_u^2 + y_u^2$. The distorted point $(x_d, y_d)$ is then:
$$
x_d = x_u + dx \\
y_d = y_u + dy
$$
The `uncalibrate` method applies this distortion model and then the intrinsic matrix $K$ to project a point from the camera frame to pixel coordinates. The `calibrate` method does the reverse, removing the distortion and intrinsic effects. Note that removing the distortion requires an iterative optimization process.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3_DS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop

In [ ]:
import gtsam
import numpy as np
from gtsam import Cal3_DS2, Point2, Pose3, PinholeCamera

## Initialization

A `Cal3_DS2` object can be initialized in several ways:

In [ ]:
# Default constructor (ideal camera with no distortion)
cal_default = Cal3_DS2()
print(f"Default constructor:\n{cal_default}\n")

# From individual parameters: fx, fy, s, u0, v0, p1, p2
fx, fy, s, u0, v0 = 1000.0, 1010.0, 0.1, 640.0, 360.0
p1, p2 = 0.001, -0.002
cal_params = Cal3_DS2(fx, fy, s, u0, v0, p1, p2)
print(f"From parameters (fx, fy, s, u0, v0, p1, p2):\n{cal_params}\n")

# From a 9-vector [fx, fy, s, u0, v0, k1, k2, p1, p2] (k1, k2 are ignored)
cal_vector = np.array([1000.0, 1010.0, 0.1, 640.0, 360.0, 0, 0, 0.001, -0.002])
cal_vec_ctor = Cal3_DS2(cal_vector)
print(f"From a 9-vector [fx, fy, s, u0, v0, k1, k2, p1, p2]:\n{cal_vec_ctor}")

## Properties

You can access all intrinsic and distortion parameters of the `Cal3_DS2` model.

In [ ]:
fx, fy, s, u0, v0 = 1000.0, 1010.0, 0.1, 640.0, 360.0
p1, p2 = 0.001, -0.002
cal = Cal3_DS2(fx, fy, s, u0, v0, p1, p2)

print("Calibration object properties:")
print(f"fx: {cal.fx()}")
print(f"fy: {cal.fy()}")
print(f"skew: {cal.skew()}")
print(f"u0: {cal.px()}")
print(f"v0: {cal.py()}")
print(f"Principal Point: {cal.principalPoint()}")
print(f"p1 (tangential distortion): {cal.p1()}")
print(f"p2 (tangential distortion): {cal.p2()}")
# Note: radial distortion parameters k1 and k2 are zero for this model
print(f"k1 (radial distortion): {cal.k1()}")
print(f"k2 (radial distortion): {cal.k2()}")

print(f"\nVector of all distortion params [k1, k2, p1, p2]: {cal.k()}")
print(f"Vector of intrinsics and distortion [fx, fy, s, u0, v0, k1, k2, p1, p2]: {cal.vector()}")
print(f"\nK matrix:\n{cal.K()}")

## Basic Operations

### `uncalibrate()`

The `uncalibrate(p)` method converts a 2D point `p` from normalized image coordinates to distorted pixel coordinates. This involves applying the tangential distortion model and then the intrinsic calibration matrix $K$.

In [ ]:
# Create a calibration model with known distortion
p1, p2 = 0.001, -0.002
cal_model_distorted = Cal3_DS2(1000, 1000, 0, 320, 240, p1, p2)

# An undistorted point in normalized coordinates
p_undistorted_norm = Point2(0.2, 0.3)

# Uncalibrate the point (apply distortion and intrinsics)
p_distorted_pixels = cal_model_distorted.uncalibrate(p_undistorted_norm)
print(f"Undistorted normalized point: {p_undistorted_norm}")
print(f"Distorted pixel point: {p_distorted_pixels}")

# Jacobian of uncalibrate wrt calibration (2x9) and point (2x2)
Dcal = np.zeros((2,9))
Dp = np.zeros((2,2))
_ = cal_model_distorted.uncalibrate(p_undistorted_norm, Dcal, Dp)
print(f"\nJacobian Dcal_uncalibrate:\n{Dcal}")
print(f"\nJacobian Dp_uncalibrate:\n{Dp}")

### `calibrate()`

The `calibrate(p)` method converts a 2D point `p` from pixel coordinates back to normalized, undistorted coordinates. This process is the inverse of `uncalibrate` and requires an iterative method to solve for the undistorted point.

In [ ]:
# Use the distorted pixel point from the previous example
p_undistorted_recovered = cal_model_distorted.calibrate(p_distorted_pixels)

print(f"Distorted pixel point: {p_distorted_pixels}")
print(f"Recovered undistorted normalized point: {p_undistorted_recovered}")

# This should be close to the original normalized point
np.testing.assert_allclose(p_undistorted_norm, p_undistorted_recovered, atol=1e-7)

# Jacobians for calibrate
Dcal_cal = np.zeros((2,9))
Dp_cal = np.zeros((2,2))
_ = cal_model_distorted.calibrate(p_distorted_pixels, Dcal_cal, Dp_cal)
print(f"\nJacobian Dcal_calibrate:\n{Dcal_cal}")
print(f"\nJacobian Dp_calibrate:\n{Dp_cal}")

## Manifold Operations

`Cal3_DS2` is an 8-dimensional manifold (since k1, k2 are fixed at 0) and supports `retract` and `localCoordinates` for optimization in the tangent space.

In [ ]:
cal_model = Cal3_DS2(1000, 1000, 0, 320, 240, 0.001, -0.002)
print("Original cal_model:")
print(cal_model)

# Define a delta in the 8-dim tangent space (fx,fy,s,u0,v0,k1,k2,p1,p2)
# Note that k1, k2 updates are ignored
delta_vec = np.array([10.0, -10.0, 0.01, 1.0, -2.0, 0, 0, 0.0005, -0.0005])
cal_retracted = cal_model.retract(delta_vec)
print(f"\nDelta vector: {delta_vec}")
print("\nRetracted cal_retracted:")
print(cal_retracted)

# Local coordinates: Find the delta between two calibrations
local_coords = cal_model.localCoordinates(cal_retracted)
print("\nLocal coordinates from cal_model to cal_retracted:")
print(local_coords)

## Relationship with `PinholeCamera`

The `Cal3_DS2` calibration can be used with a `PinholeCamera` to project 3D points into a distorted image.

In [ ]:
# Define a camera with Cal3_DS2 calibration
camera_pose = Pose3()
camera = PinholeCamera(camera_pose, cal_model_distorted)

# Project a 3D point in the camera frame
point_3d_in_camera = gtsam.Point3(0.2, 0.3, 1.0)
pixel_coords = camera.project(point_3d_in_camera)

print(f"3D point in camera frame: {point_3d_in_camera}")
print(f"Projected pixel coordinates: {pixel_coords}")

# The normalized coordinates are simply (x/z, y/z)
p_normalized = Point2(point_3d_in_camera.x() / point_3d_in_camera.z(), 
                      point_3d_in_camera.y() / point_3d_in_camera.z())
print(f"Normalized coordinates: {p_normalized}")

# Verify that this matches the uncalibrate result
pixel_coords_manual = cal_model_distorted.uncalibrate(p_normalized)
print(f"Manually uncalibrated pixel coordinates: {pixel_coords_manual}")

## Additional Resources

- [GTSAM Examples](https://github.com/borglab/gtsam/tree/develop/examples)
- Camera calibration and 3D reconstruction with OpenCV: [OpenCV Docs](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)

### Source
- [Cal3DS2.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3DS2.h)
- [Cal3DS2.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3DS2.cpp)